# TrafficGuard: Randomized Smoothing Defence
**COMP47250 · Team Software Project · Project P14 · UCD Summer 2026**

---

### What this notebook does

Implements **Randomized Smoothing** (Cohen et al., 2019) — a defence that wraps the classifier
in a noisy voting procedure: at inference time, we add many independent samples of Gaussian
noise to the input, classify each noisy copy, and take a majority vote. The resulting "smoothed"
classifier is provably robust within a certified radius around each input — not just empirically
harder to fool, but **mathematically guaranteed** to be stable up to that radius.

This is what makes it distinct from the other two defences already in the project:

| Defence | Mechanism | Where it acts | Guarantee |
|---|---|---|---|
| Spatial Smoothing | Median filter removes high-frequency noise | Preprocessing | Empirical only |
| Diffusion Purification | Reconstructs a clean image via forward/reverse diffusion | Preprocessing (generative) | Empirical only |
| **Randomized Smoothing** | **Majority vote over many Gaussian-noised copies of the input** | **Inference-time (wraps the classifier)** | **Certified radius, per Cohen et al. 2019** |

### How it works

```
Input image x
   |
   |--> add Gaussian noise (sigma = SCALE) --> copy 1 --> classify --> "Medium"
   |--> add Gaussian noise (sigma = SCALE) --> copy 2 --> classify --> "Medium"
   |--> add Gaussian noise (sigma = SCALE) --> copy 3 --> classify --> "High"
   |--> ... (SAMPLE_SIZE copies total)
   |
   v
Majority vote --> "Medium" (smoothed prediction)
   |
   v
certify() --> "this prediction is guaranteed to hold for any perturbation
               within radius R of x" (R depends on how confident the vote was)
```

### Why train with Gaussian noise augmentation too

A classifier trained only on clean images tends to perform poorly once you start feeding it
noisy copies — its decision boundaries weren't built to expect that. Cohen et al. show that
briefly fine-tuning with the *same* Gaussian noise the smoothing procedure will use at test
time (matched `sigma`) substantially improves both clean accuracy and the certified radius.
ART's `PyTorchRandomizedSmoothing.fit()` does exactly this: it adds noise to every training
batch before the forward pass, the same way `AdversarialTrainer` mixed in PGD examples for
adversarial training — just with random noise instead of a worst-case attack.

### The trade-off to know before you run this

Unlike adversarial training (which has zero extra inference cost), randomized smoothing
classifies **`SAMPLE_SIZE` noisy copies per prediction**, so a single "prediction" is actually
`SAMPLE_SIZE` forward passes. That's the cost of the certified guarantee — worth calling out
explicitly in the report alongside Diffusion Purification's ~3-5s/image cost, since both trade
inference speed for robustness, just via very different mechanisms.


---
## 0. Setup & Imports

In [ ]:
# ART: Randomized Smoothing estimator + FGSM/PGD for evaluation
from art.estimators.certification.randomized_smoothing import PyTorchRandomizedSmoothing
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

# Standard library
import os
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Visualisation
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# NOTE: certify() needs statsmodels for the Clopper-Pearson confidence bound.
# Not in backend/requirements.txt (it's only needed for this notebook) --
# pip install statsmodels if you hit an ImportError on the certification cell.


---
## 1. Configuration

### 1.1 Paths

Same convention as `spatial_smoothing.ipynb`, `diffusion_purification.ipynb`, and the
adversarial training notebook. `BASELINE_CHECKPOINT` is Soham's trained `best.pt` — we
fine-tune from it with Gaussian noise augmentation rather than training from scratch.

### 1.2 Randomized smoothing hyperparameters

- `SCALE` (sigma): standard deviation of the Gaussian noise. Bigger sigma = larger certified
  radius but lower clean accuracy — this is the main knob to tune.
- `SAMPLE_SIZE`: number of noisy copies voted over at *prediction* time. Higher = more stable
  predictions, but `SAMPLE_SIZE`x slower inference.
- `CERTIFY_N`: number of noisy copies used at *certification* time (Section 8). This is
  separate from and usually much larger than `SAMPLE_SIZE`, since certifying a tight radius
  needs a much tighter confidence bound than just picking the majority class.
- `ALPHA`: failure probability of the certification — 0.001 means the certified radius is
  correct with 99.9% confidence.

In [ ]:
# -- Paths --------------------------------------------------------------------
BASE_PATH          = Path(r'C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\data\raw\MIO-TCD-Localization')
BASELINE_CHECKPOINT = Path(r'C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\model\checkpoints\best.pt')
MANIFEST_PATH       = Path(r'data/processed/labelled_manifest.csv')
IMAGES_DIR          = Path(r'C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\data\raw\MIO-TCD-Localization\train')

ROBUST_CHECKPOINT_DIR = Path('checkpoints')
ROBUST_CHECKPOINT_DIR.mkdir(exist_ok=True)
ROBUST_CHECKPOINT_PATH = ROBUST_CHECKPOINT_DIR / 'best_randomized_smoothing.pt'

# -- Class mapping -- must match Soham's training notebook exactly ------------
CLASS_NAMES  = ['Low', 'Medium', 'High']
LABEL_MAP    = {'Low': 0, 'Medium': 1, 'High': 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_CLASSES  = 3

# -- ImageNet normalisation -- same as training --------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# -- Randomized smoothing hyperparameters --------------------------------------
SCALE           = 0.25   # sigma -- std dev of Gaussian noise (in [0,1] pixel space)
SAMPLE_SIZE     = 100    # noisy copies voted over per prediction (Sec. 7 evaluation)
CERTIFY_N       = 1000   # noisy copies used per image for certification (Sec. 8) -- slow, keep N_CERTIFY_EVAL small
ALPHA           = 0.001  # certification failure probability (99.9% confidence)
GAUSSIAN_AUG_EPOCHS = 5  # epochs of fine-tuning with matched Gaussian noise augmentation
LEARNING_RATE   = 1e-4   # small LR -- fine-tuning, not training from scratch
BATCH_SIZE      = 32


---
## 2. Load the Baseline Model

Same architecture and loading pattern as the other defence notebooks.

In [ ]:
def build_resnet18_model(num_classes=NUM_CLASSES):
    """Reconstructs the TrafficGuard ResNet18 architecture -- must match training notebook exactly."""
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def load_baseline_checkpoint(checkpoint_path, device):
    """Load Soham's trained weights."""
    if not Path(checkpoint_path).exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {checkpoint_path}\n'
            f'Make sure Soham has finished training and committed best.pt to the repo.'
        )
    model = build_resnet18_model().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    return model


baseline_model = load_baseline_checkpoint(BASELINE_CHECKPOINT, DEVICE)
baseline_model.train()
print("Baseline checkpoint loaded -- starting from Soham's converged weights.")


---
## 3. Dataset

Identical dataset class to the other notebooks.

In [ ]:
class MIOTCDDataset(Dataset):
    '''Reads the labelled_manifest.csv from Walid's pipeline. Identical across all notebooks.'''
    def __init__(self, manifest_path, split, images_dir=None, transform=None):
        df = pd.read_csv(manifest_path, dtype={'image_id': str})
        self.df = df[df['split'] == split].reset_index(drop=True)
        self.images_dir = Path(images_dir) if images_dir else None
        self.transform = transform

        unknown = set(self.df['congestion_label'].unique()) - set(LABEL_MAP)
        if unknown:
            raise ValueError(f'Unknown labels in manifest: {unknown}')
        self.df = self.df.copy()
        self.df['label_idx'] = self.df['congestion_label'].map(LABEL_MAP)

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, row):
        if self.images_dir:
            return self.images_dir / f"{row['image_id']}.jpg"
        return Path(row['image_path'])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self._resolve_path(row)
        image = Image.open(path).convert('RGB')
        label = int(row['label_idx'])
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # NOTE: no Normalize here -- ART's PyTorchRandomizedSmoothing applies (MEAN, STD)
    # internally via `preprocessing`, and expects raw [0, 1] tensors as input, same as
    # the adversarial training notebook.
])

train_dataset = MIOTCDDataset(MANIFEST_PATH, 'train', images_dir=IMAGES_DIR, transform=train_transform)
val_dataset   = MIOTCDDataset(MANIFEST_PATH, 'val',   images_dir=IMAGES_DIR, transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train : {len(train_dataset):,} images')
print(f'Val   : {len(val_dataset):,} images')


---
## 4. Wrap the Model with Randomized Smoothing

`PyTorchRandomizedSmoothing` is a drop-in replacement for `PyTorchClassifier` — same
constructor pattern you've already used twice, plus three extra arguments (`sample_size`,
`scale`, `alpha`) that control the noise/voting procedure.

In [ ]:
optimizer = optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

rs_classifier = PyTorchRandomizedSmoothing(
    model=baseline_model,
    loss=loss_fn,
    optimizer=optimizer,
    input_shape=(3, 224, 224),
    nb_classes=NUM_CLASSES,
    preprocessing=(IMAGENET_MEAN, IMAGENET_STD),
    clip_values=(0.0, 1.0),
    device_type='gpu' if DEVICE.type == 'cuda' else 'cpu',
    sample_size=SAMPLE_SIZE,
    scale=SCALE,
    alpha=ALPHA,
)
print(f'PyTorchRandomizedSmoothing ready: sigma={SCALE}, sample_size={SAMPLE_SIZE}, alpha={ALPHA}')


---
## 5. Gaussian Noise-Augmented Fine-Tuning

`rs_classifier.fit()` automatically adds `N(0, SCALE^2)` noise to every training batch before
the forward pass — no need to inject noise manually. This matches the noise level the model
will see at inference time, which is what makes the certified radius meaningful (an unmatched
sigma between training and inference gives a valid but much weaker certificate).

In [ ]:
print('Loading training set into memory for rs_classifier.fit()...')
x_train_list, y_train_list = [], []
for imgs, labels in train_loader:
    x_train_list.append(imgs.numpy())
    y_train_list.append(labels.numpy())

x_train = np.concatenate(x_train_list, axis=0)          # (N, 3, 224, 224), values in [0, 1]
y_train = np.eye(NUM_CLASSES)[np.concatenate(y_train_list, axis=0)]  # one-hot, ART convention

print(f'x_train shape: {x_train.shape}')
print(f'Starting Gaussian-augmented fine-tuning for {GAUSSIAN_AUG_EPOCHS} epochs (sigma={SCALE})...')

start = time.time()
rs_classifier.fit(x_train, y_train, nb_epochs=GAUSSIAN_AUG_EPOCHS, batch_size=BATCH_SIZE, verbose=True)
elapsed = time.time() - start

print(f'Fine-tuning complete in {elapsed/60:.1f} minutes.')


---
## 6. Save the Robust Checkpoint

In [ ]:
torch.save({
    'model_state_dict': baseline_model.state_dict(),
    'defence': 'randomized_smoothing',
    'scale': SCALE,
    'sample_size': SAMPLE_SIZE,
    'alpha': ALPHA,
    'gaussian_aug_epochs': GAUSSIAN_AUG_EPOCHS,
}, ROBUST_CHECKPOINT_PATH)

print(f'Robust checkpoint saved to: {ROBUST_CHECKPOINT_PATH}')
print('NOTE: unlike adversarial training, this checkpoint is only useful WRAPPED in')
print('PyTorchRandomizedSmoothing at inference time -- using the raw weights without the')
print('noise + majority-vote wrapper discards the smoothing procedure entirely.')


---
## 7. Evaluate — Clean vs Attacked vs Defended

Same structure as the other two defence notebooks, so results are directly comparable.
`rs_classifier.predict()` runs the full majority-vote procedure (`SAMPLE_SIZE` noisy forward
passes per image), so this cell is noticeably slower than a normal `.predict()` call — that's
expected and is the defence's core cost.

In [ ]:
baseline_model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

test_dataset = MIOTCDDataset(MANIFEST_PATH, 'test', images_dir=IMAGES_DIR, transform=eval_transform)
N_EVAL = min(100, len(test_dataset))   # smaller than the other notebooks -- smoothed predict() is SAMPLE_SIZE times slower
test_dataset.df = test_dataset.df.sample(N_EVAL, random_state=42).reset_index(drop=True)

x_list, y_list = [], []
for i in range(N_EVAL):
    img, label = test_dataset[i]
    x_list.append(img.numpy())
    y_list.append(label)
x_clean = np.stack(x_list)
y_true = np.array(y_list)

# Undefended classifier (for computing "attacked accuracy" baseline and generating attacks)
undefended_model = load_baseline_checkpoint(BASELINE_CHECKPOINT, DEVICE)
undefended_model.eval()
undefended_classifier = PyTorchClassifier(
    model=undefended_model, loss=loss_fn, input_shape=(3, 224, 224), nb_classes=NUM_CLASSES,
    preprocessing=(IMAGENET_MEAN, IMAGENET_STD), clip_values=(0.0, 1.0),
    device_type='gpu' if DEVICE.type == 'cuda' else 'cpu',
)

EPSILON = 0.10  # matches Payal's FGSM eval epsilon and the adversarial training notebook

results = {}
for attack_name, attack_cls, attack_kwargs in [
    ('FGSM', FastGradientMethod, dict(eps=EPSILON)),
    ('PGD',  ProjectedGradientDescent, dict(eps=EPSILON, eps_step=0.02, max_iter=10)),
]:
    attack = attack_cls(estimator=undefended_classifier, **attack_kwargs)
    x_adv = attack.generate(x=x_clean)

    clean_preds = np.argmax(undefended_classifier.predict(x_clean), axis=1)
    adv_preds   = np.argmax(undefended_classifier.predict(x_adv), axis=1)

    # rs_classifier.predict() returns one-hot smoothed predictions (with possible abstentions
    # encoded as all-zero rows when the vote isn't confident enough)
    defended_onehot = rs_classifier.predict(x_adv, batch_size=BATCH_SIZE)
    abstained = np.all(defended_onehot == 0, axis=1)
    defended_preds = np.argmax(defended_onehot, axis=1)

    clean_acc    = np.mean(clean_preds == y_true)
    adv_acc      = np.mean(adv_preds == y_true)
    defended_acc = np.mean((defended_preds == y_true) & ~abstained)  # abstentions count as incorrect
    asr          = np.mean(adv_preds != clean_preds)

    results[attack_name] = dict(
        clean_acc=clean_acc, adv_acc=adv_acc, defended_acc=defended_acc,
        asr=asr, abstain_rate=abstained.mean(),
    )

    print(f'=== {attack_name} (eps={EPSILON}) ===')
    print(f'  Clean accuracy            : {clean_acc:.2%}')
    print(f'  Attacked accuracy         : {adv_acc:.2%}   (down {clean_acc - adv_acc:.2%})')
    print(f'  Defended accuracy         : {defended_acc:.2%}   (up {defended_acc - adv_acc:.2%} recovery)')
    print(f'  Attack Success Rate (ASR) : {asr:.2%}')
    print(f'  Abstention rate           : {abstained.mean():.2%}')
    print()


### 7.1 Confusion matrix — smoothed classifier on PGD-attacked test images

In [ ]:
x_adv_pgd = ProjectedGradientDescent(
    estimator=undefended_classifier, eps=EPSILON, eps_step=0.02, max_iter=10
).generate(x=x_clean)

defended_onehot_pgd = rs_classifier.predict(x_adv_pgd, batch_size=BATCH_SIZE)
defended_preds_pgd = np.argmax(defended_onehot_pgd, axis=1)

cm = confusion_matrix(y_true, defended_preds_pgd)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Randomized Smoothing Classifier\nPredictions on PGD-Attacked Test Images')
plt.tight_layout()
plt.savefig('randomized_smoothing_confusion_matrix.png', dpi=120)
plt.show()

print(classification_report(y_true, defended_preds_pgd, target_names=CLASS_NAMES))


---
## 8. Certification — the Headline Result for This Defence

This is the one number the other two defences can't produce: a **certified radius** per image.
`certify()` guarantees that the prediction cannot change for *any* perturbation (not just FGSM
or PGD — any L2 perturbation at all) smaller than the returned radius, with `1 - ALPHA`
confidence. This is much slower than a normal prediction (`CERTIFY_N` noisy forward passes per
image), so we run it on a small subset — the point is to report a median certified radius and
a certified-accuracy-at-radius-R curve for the write-up, not to certify every image.

In [ ]:
N_CERTIFY_EVAL = min(20, N_EVAL)   # certification is expensive -- keep this small
x_certify = x_clean[:N_CERTIFY_EVAL]
y_certify = y_true[:N_CERTIFY_EVAL]

print(f'Certifying {N_CERTIFY_EVAL} clean test images with n={CERTIFY_N} samples each...')
print(f'(This is {N_CERTIFY_EVAL} x {CERTIFY_N} = {N_CERTIFY_EVAL * CERTIFY_N:,} forward passes -- may take a while)')

start = time.time()
certified_preds, certified_radii = rs_classifier.certify(x_certify, n=CERTIFY_N, batch_size=BATCH_SIZE)
elapsed = time.time() - start
print(f'Certification complete in {elapsed/60:.1f} minutes.')

correct_and_certified = (certified_preds == y_certify) & (certified_radii > 0)

print()
print(f'Median certified radius (correct predictions) : {np.median(certified_radii[correct_and_certified]):.4f}')
print(f'Mean certified radius (correct predictions)    : {np.mean(certified_radii[correct_and_certified]):.4f}')
print(f'Certified accuracy at r > 0                    : {correct_and_certified.mean():.2%}')


### 8.1 Certified accuracy vs radius curve

Standard plot from the Cohen et al. paper — the fraction of images still correctly and certifiably classified as the required robustness radius increases.

In [ ]:
radius_thresholds = np.linspace(0, certified_radii.max() if certified_radii.max() > 0 else 0.5, 20)
certified_acc_at_r = [
    np.mean((certified_preds == y_certify) & (certified_radii >= r))
    for r in radius_thresholds
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(radius_thresholds, certified_acc_at_r, marker='o', color='#1976D2')
ax.set_xlabel('Certified radius r (L2, pixel space [0,1])')
ax.set_ylabel('Certified accuracy')
ax.set_title(f'TrafficGuard -- Certified Accuracy vs Radius\n(sigma={SCALE}, n={CERTIFY_N}, {N_CERTIFY_EVAL} images)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('certified_accuracy_vs_radius.png', dpi=120)
plt.show()


---
## 9. Summary — All Three Defences Side by Side

Fill in the Spatial Smoothing and Diffusion Purification numbers from their respective
notebooks to complete this table for the report.

In [ ]:
print('TRAFFICGUARD -- RANDOMIZED SMOOTHING SUMMARY')
print('=' * 58)
print(f'Defence            : Randomized Smoothing')
print(f'Reference          : Cohen et al., ICML 2019')
print(f'Noise sigma        : {SCALE}')
print(f'Sample size (vote) : {SAMPLE_SIZE}')
print(f'Certify n          : {CERTIFY_N}')
print(f'Gaussian aug epochs: {GAUSSIAN_AUG_EPOCHS}')
print(f'Eval images        : {N_EVAL}')
print()
for attack_name, r in results.items():
    print(f'[{attack_name}]  clean {r["clean_acc"]:.2%}  ->  attacked {r["adv_acc"]:.2%}  ->  defended {r["defended_acc"]:.2%}'
          f'   (recovery: +{r["defended_acc"] - r["adv_acc"]:.2%}, abstain: {r["abstain_rate"]:.2%})')
print()
print(f'Median certified radius : {np.median(certified_radii[correct_and_certified]):.4f}')
print(f'Certified accuracy at r>0 : {correct_and_certified.mean():.2%}')
print()
print('Key design decisions:')
print('  - Fine-tuned with matched Gaussian noise (sigma) rather than training from scratch,')
print('    per Cohen et al.\'s recommendation for best accuracy/certification trade-off')
print('  - Evaluated empirically against FGSM and PGD, same as the other two defences,')
print('    PLUS a formal certified radius that neither of them can produce')
print('  - Certification run on a small subset (N_CERTIFY_EVAL) due to its cost --')
print('    scale up for final report numbers if time allows')
print()
print('Limitations to note in final report:')
print('  - Inference cost scales with SAMPLE_SIZE -- a single "prediction" is SAMPLE_SIZE')
print('    forward passes, which matters a lot for the real-time dashboard use case')
print('  - Larger sigma gives a bigger certified radius but costs clean accuracy -- this')
print('    notebook uses sigma=0.25 as a starting point; sweep sigma if time allows')
print('  - The certified guarantee covers L2 perturbations of any kind, which is broader')
print('    than "robust to FGSM/PGD specifically" -- but it is also a probabilistic')
print('    guarantee (via ALPHA), not an absolute one')


---
## 10. Integration Notes for the Dashboard

This defence needs a **different integration shape** than the other two:

- Not a preprocessing step like Spatial Smoothing or Diffusion Purification (no image
  transformation to insert before `classify()`).
- Not a drop-in checkpoint swap like Adversarial Training either — the checkpoint alone is
  just a Gaussian-noise-robust ResNet18; the *defence* is the `PyTorchRandomizedSmoothing`
  wrapper and its `.predict()` majority-vote procedure around it.
- In `backend/ml.py`, the model-loading code needs to construct a
  `PyTorchRandomizedSmoothing` instance (as in Section 4) around the loaded checkpoint, and
  `app.py`'s prediction endpoint needs to call `.predict()` on that wrapper rather than the
  raw model — expect `SAMPLE_SIZE` times the latency of a normal prediction, so this is the
  one defence where per-request latency should be benchmarked before deciding whether it's
  viable for the live dashboard vs. an offline/batch analysis mode.
- `certify()` is almost certainly too slow for live requests (`CERTIFY_N` forward passes per
  image) — it's best used for report figures like Section 8, not the running dashboard.
